In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Load the saved clusters from the specified iterations
initial_clusters_old = np.load('data_array.npy')
labels_old = initial_clusters[:, 11]

# Extract x, y columns
x = initial_clusters_old[:, 0]
y = initial_clusters_old[:, 1]

# Boolean mask: keep all rows EXCEPT the one with x=102 and y=120
mask_old = ~((x == 102) & (y == 120))

# Apply mask to remove the row
initial_clusters = initial_clusters_old[mask_old]
labels= labels_old[mask_old]

print(np.unique(labels, return_counts=True))

# Set labels to 4 where the second column is below 126
# labels[initial_clusters[:, 1] > 126] = 4  # Set label to 4 where condition is met

# Create a mask for labels 7 and 4
mask = (labels == 7) | (labels == 4)
xyz_initial = initial_clusters[mask, :3]
labels_initial = labels[mask]

# Set plot limits and grid size
x_limits = (100, 200)
y_limits = (60, 136)
pixel_size_mm = 2

# Create a 2x2 subplot for double-column format
fig, axs = plt.subplots(1, 2, figsize=(12, 8), dpi=300, constrained_layout=True)

# Function to add rectangle patches for each point
def add_rectangles(ax, xyz_data, labels, cmap):
    for (x, y, _), label in zip(xyz_data, labels):
        color = cmap(int(label) % cmap.N)
        rect = patches.Rectangle((x, y), pixel_size_mm, pixel_size_mm, linewidth=0.5,
                                 edgecolor='none', facecolor=color, alpha=0.7)
        ax.add_patch(rect)

# Function to set custom grid and sparse Y-axis tick labels
def set_custom_grid(ax):
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    
    # Set grid ticks every 2 mm on both axes
    ax.set_xticks(np.arange(x_limits[0], x_limits[1] + 1, pixel_size_mm), minor=False)
    ax.set_yticks(np.arange(y_limits[0], y_limits[1] + 1, pixel_size_mm), minor=True)
    ax.set_xticks(np.arange(x_limits[0], x_limits[1] + 1, pixel_size_mm), minor=True)
    ax.set_yticks(np.arange(y_limits[0], y_limits[1] + 1, 8), minor=False)
    ax.set_xticks(np.arange(x_limits[0], x_limits[1] + 1, 8), minor=False)
    
    # Display grid
    ax.grid(which="both", color="gray", linestyle="--", linewidth=0.5)

    # Label Y-axis major ticks every 8 mm
    y_labels = np.arange(y_limits[0], y_limits[1] + 1, 8)
    ax.set_yticklabels(y_labels)

    # Add thicker grid lines between Y = 122 and Y = 132
    ax.axhline(y=122, color='black', linewidth=2, linestyle='--')  # Thicker grid line at Y=122
    ax.axhline(y=132, color='black', linewidth=2, linestyle='--')  # Thicker grid line at Y=132

# Mean vector [x, y, z] (the starting point)
mean_vector = np.array([151.25, 94.75, 260.63])

# Direction vector [dx, dy, dz] (direction from the mean)
direction_vector = np.array([0.68, -0.66, 0.32])

# Set a scaling factor to determine the length of the arrow (you can adjust this value)
scaling_factor = 10  # This controls how long the direction vector is on the plot

# Calculate the end point by adding the scaled direction vector to the mean vector
end_point = mean_vector[:2] + scaling_factor * direction_vector[:2]

# Define a colormap
# cmap = plt.cm.get_cmap("viridis", len(np.unique(labels_initial)))

# Load the closest points
closest_points = np.load('closest_points.npy')

# Extract x, y, and z columns (assuming columns are x, y, z)
x_closest_points = closest_points[:, 0]
y_closest_points = closest_points[:, 1]
z_closest_points = closest_points[:, 2]

# Normalize the direction vector to unit length for projections
direction_unit_vector = direction_vector / np.linalg.norm(direction_vector)

# Calculate the projections of the points onto the direction and negative direction vectors
projections_pos = np.dot(closest_points - mean_vector, direction_unit_vector)
projections_neg = np.dot(closest_points - mean_vector, -direction_unit_vector)

# Find the points with the maximum projection in the positive direction
max_pos_idx = np.argmax(projections_pos)
max_neg_idx = np.argmax(projections_neg)

# Get the extreme points
extreme_point_pos = closest_points[max_pos_idx]
extreme_point_neg = closest_points[max_neg_idx]

# New intersection point
intersection_point = np.array([115.07, 129.64, 243.29])

cmap = plt.cm.get_cmap("Set1", len(np.unique(labels_initial)))

# Plot Initial Clusters
set_custom_grid(axs[0])
add_rectangles(axs[0], xyz_initial, labels_initial, cmap)
axs[0].set_xlabel('X [mm]', fontname='Arial', fontsize=16, fontweight='bold')
axs[0].set_ylabel('Y [mm]', fontname='Arial', fontsize=16, fontweight='bold')
axs[0].text(0.05, 0.95, '(a)', transform=axs[0].transAxes, fontsize=16, verticalalignment='top')

# Plot the mean point (starting point) and direction vector using ax.arrow
axs[0].arrow(mean_vector[0], mean_vector[1], direction_vector[0] * scaling_factor, direction_vector[1] * scaling_factor, 
          head_width=4, head_length=5, fc='r', ec='r')

# Add text label for direction vector d_k
axs[0].text(mean_vector[0] + direction_vector[0] * scaling_factor * 0.7, 
            mean_vector[1] + direction_vector[1] * scaling_factor * 0.7, 
            '$d_k$', color='r', fontsize=20, fontweight='bold')

# Plot the starting point (mean vector) for reference
axs[0].plot(mean_vector[0], mean_vector[1], 'bo', label='Mean Vector')
axs[0].text(mean_vector[0] + 2, mean_vector[1], r'$\mu_k$', fontsize=20, verticalalignment='bottom', c='b')
axs[0].scatter(x_closest_points, y_closest_points, c='g', label='Closest Points', s=10)

# Plot the extreme points with larger "X" markers
axs[0].plot(extreme_point_pos[0], extreme_point_pos[1], 'gx', label='Extreme Point (Positive)', markersize=15)
axs[0].plot(extreme_point_neg[0], extreme_point_neg[1], 'bx', label='Extreme Point (Negative)', markersize=15)

# Add text labels for extreme points
axs[0].text(extreme_point_pos[0] + 3, extreme_point_pos[1], '$e_k$', color='g', fontsize=20, fontweight='bold')  # Position to the right of positive extreme
axs[0].text(extreme_point_neg[0] + 3, extreme_point_neg[1], '$s_k$', color='b', fontsize=20, fontweight='bold')  # Position to the right of negative extreme

# Plot the intersection point with a unique marker (e.g., red star)
axs[0].plot(intersection_point[0], intersection_point[1], 'g*', label='Intersection Point', markersize=12)
# Add text label for intersection point v_k
axs[0].text(intersection_point[0] + 2, intersection_point[1] + 3,  # Adjusting position slightly
            '$v_k$', color='g', fontsize=20, fontweight='bold')

# Load the charge profiles
charge_profile_x = np.load('charge_profile_x.npy')
charge_profile_y = np.load('charge_profile_y.npy')
charge_profile_x_s = np.load('charge_profile_x_s.npy')
charge_profile_y_s = np.load('charge_profile_y_s.npy')

# Load special points
special_points = np.array([[35.50, 1241.08], [99.77, 353.97]])

# Normalize charge_profile_y separately
y_min_1, y_max_1 = np.min(charge_profile_y), np.max(charge_profile_y)
charge_profile_y_norm = (charge_profile_y - y_min_1) / (y_max_1 - y_min_1)

# Normalize charge_profile_y_s separately
y_min_2, y_max_2 = np.min(charge_profile_y_s), np.max(charge_profile_y_s)
charge_profile_y_s_norm = (charge_profile_y_s - y_min_2) / (y_max_2 - y_min_2)

# Normalize special points based on max of charge_profile_y_s
special_points[:, 1] = special_points[:, 1] 

# Plot charge_profile_y vs. charge_profile_x
axs[1].plot(charge_profile_x, charge_profile_y, 
        color='b', linestyle='-', marker='o', markersize=3)

# Plot charge_profile_y_s vs. charge_profile_x_s
axs[1].plot(charge_profile_x_s, charge_profile_y_s, 
        color='r', linestyle='-', marker='s', markersize=3)

# Plot the two specific points with normalized y-values
axs[1].scatter(special_points[0, 0], special_points[0, 1], color='g', marker='X', s=300)
axs[1].text(special_points[0, 0] + 5, special_points[0, 1], 'Q_max', fontsize=12, color='g')
axs[1].scatter(special_points[1, 0], special_points[1, 1], color='k', marker='X', s=300)
axs[1].text(special_points[1, 0] + 5, special_points[1, 1], r'$f^*Q_{\max}$', fontsize=12, color='k')

# Labels and legend
axs[1].set_xlabel(r'Distance from $v_k$ [mm]', fontsize=16, fontweight='bold')
axs[1].set_ylabel(r'$\mathbf{Q}$ [arb]', fontsize=16, fontweight='bold')
axs[1].legend()
axs[1].grid(True, linestyle="--", linewidth=0.5)

axs[1].text(0.05, 0.95, '(b)', transform=axs[1].transAxes, fontsize=16, verticalalignment='top')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('h_fig8.png', dpi=300, bbox_inches='tight')
plt.show()
# fig.savefig("reaction_vertex.eps", format='eps', dpi=300)

NameError: name 'initial_clusters' is not defined